[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/llm-engineering-certified/notebooks/day-06-text-splitting.ipynb#scrollTo=1a2b3c4d)

---
# Day 6 · Text Splitting Strategies — Chunks, Tokens, and Overlap
**certified-journeys / llm-engineering-certified** · Day 6 · Chunking

> **Goal for today:** Split a corpus of Documents into retrieval-ready chunks using character-based and token-based splitters, compare their trade-offs quantitatively, and visualise chunk size distributions.


In [ ]:
%pip install -q langchain langchain-community langchain-text-splitters pypdf tiktoken


## Step 1 · Why Split Documents at All?

LLMs have a finite context window. A 200-page PDF can't fit in a single prompt. Text splitters solve this by breaking `Document` objects into **chunks** — smaller `Document` objects that preserve the original `metadata`.

### The three levers

| Lever | What it controls | Rule of thumb |
|---|---|---|
| `chunk_size` | Maximum chunk length (chars or tokens) | 1000 chars / 256 tokens to start |
| `chunk_overlap` | How many chars/tokens repeat in adjacent chunks | 10–20% of chunk_size |
| Separator | Where the splitter is allowed to cut | Prefer `\n\n` > `\n` > ` ` |

Official concept page: https://python.langchain.com/docs/concepts/text_splitters/


In [ ]:
# Imports shared across the notebook
import os, urllib.request, pathlib, statistics, math
from collections import Counter
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader


## Step 2 · Load a PDF for Splitting

We'll reuse the same public PDF from Day 5. Loading produces one `Document` per page — the starting material for all splitter comparisons.


In [ ]:
PDF_URL  = "https://www.w3.org/WAI/WCAG21/wcag21-diff.pdf"
PDF_PATH = "/tmp/sample.pdf"

if not os.path.exists(PDF_PATH):
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
    print("Downloaded PDF")
else:
    print("PDF already cached")

loader = PyPDFLoader(PDF_PATH)
pages  = loader.load()

# Merge all pages into a single long document for splitting experiments
full_text = "\n\n".join(p.page_content for p in pages)
source_doc = Document(page_content=full_text, metadata={"source": "wcag21-diff.pdf"})

print(f"Pages loaded        : {len(pages)}")
print(f"Total characters    : {len(full_text):,}")
print(f"Approx. word count  : {len(full_text.split()):,}")


### What just happened?

- We joined all PDF pages into one large `Document` — this simulates a real ingestion scenario where page boundaries shouldn't control chunk boundaries.
- **Character count** is the unit `RecursiveCharacterTextSplitter` uses; **token count** is what `TokenTextSplitter` uses.
- The same source document will be passed to every splitter so comparisons are apples-to-apples.
- `metadata` is copied to every chunk automatically — the `source` key will survive splitting.


## Step 3 · RecursiveCharacterTextSplitter (chunk_size=1000, overlap=200)

`RecursiveCharacterTextSplitter` tries to split on `["\n\n", "\n", " ", ""]` in order — it keeps semantically related text together by preferring paragraph breaks first.

```python
RecursiveCharacterTextSplitter(
    chunk_size=1000,     # max characters per chunk
    chunk_overlap=200,   # characters repeated at chunk boundaries
    separators=["\n\n", "\n", " ", ""],
)
```

This is **the right default** for most use cases.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

char_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""],  # priority order
)

char_chunks = char_splitter.split_documents([source_doc])

char_lengths = [len(c.page_content) for c in char_chunks]

print(f"Chunk count         : {len(char_chunks)}")
print(f"Min length (chars)  : {min(char_lengths)}")
print(f"Max length (chars)  : {max(char_lengths)}")
print(f"Mean length (chars) : {statistics.mean(char_lengths):.0f}")
print(f"Median (chars)      : {statistics.median(char_lengths):.0f}")
print(f"\nChunk 0 metadata    : {char_chunks[0].metadata}")
print(f"Chunk 0 preview     : {char_chunks[0].page_content[:200]!r}")


### What just happened?

- `split_documents` returns a new `list[Document]` where **every chunk inherits the original metadata**.
- Some chunks are shorter than `chunk_size=1000` — that's normal when a paragraph is shorter than the limit.
- **Overlap** means adjacent chunks share 200 characters. This prevents a sentence that straddles a boundary from being lost in retrieval.
- **Max length can exceed `chunk_size`** if a single word or token is longer than the limit — extremely rare in prose.


## Step 4 · TokenTextSplitter with tiktoken (chunk_size=256 tokens)

`TokenTextSplitter` counts tokens using `tiktoken` (OpenAI's tokeniser). This is the right splitter when you need **predictable context-window consumption** — e.g., when using `text-embedding-3-small` which has an 8191 token limit.

```python
TokenTextSplitter(
    encoding_name="cl100k_base",  # GPT-4 / embedding tokenizer
    chunk_size=256,               # tokens per chunk
    chunk_overlap=32,             # tokens of overlap
)
```


In [ ]:
from langchain_text_splitters import TokenTextSplitter
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")  # GPT-4 / text-embedding-3-* tokenizer

token_splitter = TokenTextSplitter(
    encoding_name="cl100k_base",
    chunk_size=256,    # tokens
    chunk_overlap=32,  # ~12% overlap
)

token_chunks = token_splitter.split_documents([source_doc])

# Count tokens per chunk for accurate comparison
token_lengths = [len(enc.encode(c.page_content)) for c in token_chunks]
char_equiv    = [len(c.page_content) for c in token_chunks]

print(f"Chunk count           : {len(token_chunks)}")
print(f"Min tokens per chunk  : {min(token_lengths)}")
print(f"Max tokens per chunk  : {max(token_lengths)}")
print(f"Mean tokens per chunk : {statistics.mean(token_lengths):.1f}")
print(f"Mean chars per chunk  : {statistics.mean(char_equiv):.0f}")
print(f"\nChunk 0 token count   : {len(enc.encode(token_chunks[0].page_content))}")
print(f"Chunk 0 preview       : {token_chunks[0].page_content[:200]!r}")


### What just happened?

- `TokenTextSplitter` produces **more chunks** than `RecursiveCharacterTextSplitter` at the same rough context — because 256 tokens ≈ 200–300 words, which is smaller than 1000 characters of prose.
- Token counts are **deterministic** for a given tokeniser — useful for budgeting API calls.
- **Tokens ≠ characters**: English prose averages ~4 chars/token; code is often 2–3 chars/token.
- Use `cl100k_base` for OpenAI models (GPT-3.5/4, embeddings); use `o200k_base` for GPT-4o.


## Step 5 · Compare the Two Strategies

A quantitative comparison reveals which strategy better suits your use case.


In [ ]:
# Side-by-side statistical comparison
def stats(lengths, label):
    return {
        "splitter"  : label,
        "count"     : len(lengths),
        "min"       : min(lengths),
        "max"       : max(lengths),
        "mean"      : round(statistics.mean(lengths)),
        "stdev"     : round(statistics.stdev(lengths)),
        "median"    : round(statistics.median(lengths)),
    }

char_stats  = stats(char_lengths,  "RecursiveChar (1000 chars)")
token_stats = stats(token_lengths, "TokenText     (256 tokens)")

header = f"{'Metric':<12}  {'RecursiveChar':>20}  {'TokenText':>20}"
print(header)
print("-" * len(header))
for key in ["count", "min", "max", "mean", "stdev", "median"]:
    print(f"{key:<12}  {char_stats[key]:>20}  {token_stats[key]:>20}")

print("\nBoundary quality check — first 3 char chunks:")
for i, c in enumerate(char_chunks[:3]):
    snippet = c.page_content[:80].replace("\n", "↵")
    print(f"  [{i}] {snippet!r}")


### What just happened?

- **`RecursiveChar`** has **lower stdev** — it respects paragraph breaks, so most chunks are close to 1000 chars.
- **`TokenText`** has more uniform token counts but may cut in the middle of a sentence.
- **Higher chunk count** from `TokenText` means more vector store entries and more retrieval candidates — trade-off: higher recall, higher cost.
- **When to choose `TokenText`:** embedding APIs bill per token, you need hard context-window guarantees, or you're using a reranker that prefers short passages.
- **When to choose `RecursiveChar`:** default RAG pipelines, human-readable chunks, markdown/prose documents.


## Step 6 · MarkdownHeaderTextSplitter — Preserve Headers

`MarkdownHeaderTextSplitter` splits on `#`, `##`, `###` headers and **promotes header text into metadata**. This is ideal for README files, documentation, and wikis.

Docs: https://python.langchain.com/docs/how_to/markdown_header_metadata_splitter/


In [ ]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

# Sample README-style markdown document
readme_text = """
# LangChain Overview

LangChain is a framework for developing applications powered by language models.
It provides abstractions for building chains, agents, and RAG pipelines.

## Installation

Install the core package and community integrations:

```bash
pip install langchain langchain-community
```

## Document Loaders

Document loaders convert raw files into Document objects.
Supported sources include PDFs, web pages, databases, and cloud storage.
Always enrich metadata before ingesting into a vector store.

### PyPDFLoader

PyPDFLoader splits PDFs by page. Each page becomes one Document with
a `page` field in metadata.

### WebBaseLoader

WebBaseLoader fetches a URL and strips HTML via BeautifulSoup.
Use SoupStrainer to target only the relevant article body.

## Text Splitters

Text splitters break large Documents into retrieval-ready chunks.
The two most common splitters are RecursiveCharacterTextSplitter
and TokenTextSplitter.
"""

# Define which header levels to split on and their metadata key names
headers_to_split = [
    ("#",  "h1"),
    ("##", "h2"),
    ("###","h3"),
]

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split,
    strip_headers=False,     # keep header text in page_content too
)

md_chunks = md_splitter.split_text(readme_text)

print(f"Chunks produced: {len(md_chunks)}")
for i, chunk in enumerate(md_chunks):
    print(f"\n[Chunk {i}]")
    print(f"  metadata    : {chunk.metadata}")
    print(f"  content     : {chunk.page_content[:120].strip()!r}")


### What just happened?

- Each chunk's `metadata` now contains `h1`, `h2`, and/or `h3` keys with the **header text as value**.
- This means you can **filter by section** in a vector store: `where h2 == 'Document Loaders'`.
- `strip_headers=False` keeps the header in `page_content` — useful when the LLM needs context about which section it's reading.
- After header-splitting, you can **chain with `RecursiveCharacterTextSplitter`** if individual sections are still too long.


## Step 7 · Chunk Length Histogram

A histogram reveals **outlier-sized chunks** — tiny stubs (headers with no body) and giant blocks (dense tables). Both degrade retrieval quality.


In [ ]:
def text_histogram(lengths: list, label: str, bins: int = 10) -> None:
    """Print an ASCII histogram of chunk lengths."""
    if not lengths:
        print("No data.")
        return

    lo, hi  = min(lengths), max(lengths)
    width   = math.ceil((hi - lo + 1) / bins) or 1
    buckets = Counter()
    for n in lengths:
        bucket = ((n - lo) // width) * width + lo
        buckets[bucket] += 1

    max_count = max(buckets.values())
    bar_width  = 40

    print(f"\n=== {label} — {len(lengths)} chunks ===")
    for bucket in sorted(buckets):
        count = buckets[bucket]
        bar   = "█" * round(count / max_count * bar_width)
        label_str = f"{bucket:>5}–{bucket + width - 1:<5}"
        print(f"  {label_str} | {bar:<{bar_width}} {count}")

    # Flag outliers: chunks < 10% or > 110% of the median
    med = statistics.median(lengths)
    tiny  = [n for n in lengths if n < med * 0.10]
    giant = [n for n in lengths if n > med * 1.10]
    print(f"  Median: {med:.0f} | Tiny (<10% median): {len(tiny)} | Giant (>110% median): {len(giant)}")

text_histogram(char_lengths,  "RecursiveCharTextSplitter — chars")
text_histogram(token_lengths, "TokenTextSplitter — tokens")


### What just happened?

- The **RecursiveChar histogram** should be left-skewed — most chunks near 1000, a tail of smaller ones where paragraph breaks forced an early cut.
- The **Token histogram** should be tighter — `TokenTextSplitter` fills every chunk to its limit more aggressively.
- **Tiny chunks** (under 10% of median) are usually section headers or single sentences — they dilute retrieval results with low-information content.
- **Giant chunks** can happen when `chunk_size` is set too high relative to the separator frequency — consider reducing `chunk_size` or adding a `\n` separator.


In [ ]:
# Demonstrate chaining MarkdownHeaderTextSplitter → RecursiveCharacterTextSplitter
# for sections that are individually too long
from langchain_text_splitters import RecursiveCharacterTextSplitter

# First: split by headers
md_header_chunks = md_splitter.split_text(readme_text)

# Second: split any section longer than 300 chars by character
char_splitter_small = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
)

final_chunks = char_splitter_small.split_documents(md_header_chunks)

print(f"After header split  : {len(md_header_chunks)} chunks")
print(f"After char re-split : {len(final_chunks)} chunks")
print("\nSample final chunk:")
print(f"  metadata : {final_chunks[0].metadata}")
print(f"  content  : {final_chunks[0].page_content[:200]!r}")


### What just happened?

- Chaining splitters is the canonical pattern for structured documents: **semantic split first, size split second**.
- Header metadata survives the second split — every final chunk still knows which `h1`/`h2` section it belongs to.
- This two-stage approach gives you **semantic coherence** (header boundaries respected) **and** size guarantees (no chunk exceeds 300 chars).


In [ ]:
# Bonus: count tokens in char-split chunks to understand token budget
char_token_counts = [len(enc.encode(c.page_content)) for c in char_chunks]

print("Token budget summary for RecursiveChar chunks (chunk_size=1000 chars):")
print(f"  Mean tokens/chunk : {statistics.mean(char_token_counts):.1f}")
print(f"  Max tokens/chunk  : {max(char_token_counts)}")
print(f"  Total tokens      : {sum(char_token_counts):,}")

# Estimate embedding cost at $0.02 / 1M tokens (text-embedding-3-small)
cost_usd = sum(char_token_counts) / 1_000_000 * 0.02
print(f"  Estimated embed cost (text-embedding-3-small): ${cost_usd:.5f}")


### What just happened?

- Even though `RecursiveChar` splits by characters, you can still count tokens to **predict embedding cost**.
- `text-embedding-3-small` is priced per token, not per character — always count tokens before bulk ingestion.
- The total token count lets you spot immediately if a corpus is too large for a single embedding batch.
- **Rule of thumb:** 1000 chars ≈ 200–250 tokens for English prose. Use this to translate chunk_size requirements between char and token units.


In [ ]:
# Challenge: Build a smart_split function
#
# Implement smart_split(docs, strategy) where:
#   - strategy="char"  → RecursiveCharacterTextSplitter(chunk_size=1000, overlap=200)
#   - strategy="token" → TokenTextSplitter(chunk_size=256, overlap=32)
#   - strategy="auto"  → use "token" if any doc has >50k chars, else "char"
#
# The function should:
#   1. Select the right splitter based on strategy
#   2. Split the documents
#   3. Add a "chunk_strategy" key to each chunk's metadata
#   4. Return (chunks, stats_dict) where stats_dict has: count, mean_len, max_len
#
# Scaffold:
# def smart_split(docs, strategy="char"):
#     # YOUR CODE HERE: pick splitter
#     splitter = ...
#
#     # YOUR CODE HERE: split
#     chunks = ...
#
#     # YOUR CODE HERE: enrich metadata
#     for chunk in chunks:
#         chunk.metadata["chunk_strategy"] = ...
#
#     # YOUR CODE HERE: compute stats
#     stats = ...
#
#     return chunks, stats
#
# # Test it
# chunks_c, stats_c = smart_split([source_doc], strategy="char")
# chunks_t, stats_t = smart_split([source_doc], strategy="token")
# chunks_a, stats_a = smart_split([source_doc], strategy="auto")
# print(stats_c, stats_t, stats_a)


---
## Day 6 key concepts recap

| Concept | What to remember |
|---|---|
| `RecursiveCharacterTextSplitter` | Default choice; respects `\n\n` paragraph breaks; `chunk_size` in chars |
| `TokenTextSplitter` | Guarantees token budget; use `cl100k_base` for OpenAI models |
| `MarkdownHeaderTextSplitter` | Promotes headers to `metadata`; enables section-level filtered retrieval |
| `chunk_overlap` | 10–20% of `chunk_size`; prevents context loss at boundaries |
| Histogram analysis | Spot tiny stubs and giant blocks — both hurt retrieval quality |
| Chaining splitters | Header-split → char-split for structured docs with long sections |
| Token cost estimation | Count tokens before bulk embedding to avoid surprise API bills |

> **Tip:** Start with chunk_size=1000 and overlap=200. If retrieval misses context that clearly lives in one paragraph, reduce chunk_size — don't increase overlap blindly.

---
## What's next
**Day 7** → Vector Stores and Embeddings — embed your chunks and store them in a vector database for semantic similarity search.

Mark Day 6 complete in your [tracker](../index.html).
